In [44]:
from symmer.operators.base import PauliwordOp, QuantumState
import h5py
import numpy as np 
from pyscf import scf
import math

In [45]:
chkfile = "O2.h5"

# load chemistry data
mol_load = scf.chkfile.load_mol(chkfile)
scf_load = scf.ROHF(mol_load) 
scf_load.__dict__.update(scf.chkfile.load(chkfile, 'scf')) 

# print chemistry details (note other info can be accessed in these objects see PySCF for further details.)
print(f"n spin orbitals: {2*mol_load.nao}")
print(f"basis: {mol_load.basis}")
xyz_str = mol_load.tostring(format="xyz")
print(f"geometry:\n\n{xyz_str}")
print(f"unit: {mol_load.unit}")
print(f"initaliziation: {scf_load.init_guess}")
E_rohf = scf_load.e_tot
print(f"e_rohf: {E_rohf}")

n spin orbitals: 20
basis: STO-3G
geometry:

2
XYZ from PySCF
O           0.00000000        0.00000000        0.00000000
O           0.00000000        0.00000000        2.25000000
unit: angstrom
initaliziation: minao
e_rohf: -147.47439914439556


In [46]:
serach_space = math.comb(mol_load.nao, mol_load.nelec[0]) * math.comb(mol_load.nao, mol_load.nelec[1]) 
serach_space

1200

In [47]:
with h5py.File("O2_data.h5", "r") as infile:
    # print(infile.keys())
    H = PauliwordOp(infile["H_symp"][()], infile["H_coeff"][()])

    hf_psi = QuantumState(infile["hf_state_mat"][()], infile["hf_coeff_vec"][()])
    e_rohf = infile["e_rohf"][()]
    
    fci_psi = QuantumState(infile["fci_state_mat"][()], infile["fci_coeff_vec"][()])
    e_fci = infile["e_fci"][()]

In [48]:
E_hf = H.expval(hf_psi).real
E_hf, e_rohf

(np.float64(-147.4743991443956), np.float64(-147.47439914439556))

In [49]:
E_fci = H.expval(fci_psi).real
E_fci, e_fci

(np.float64(-147.61021835281983), np.float64(-147.610218352825))

In [50]:
print(f"energy ROHF: {E_hf.real:0.9f}")
print(f"energy fci : {E_fci.real:0.9f}")

energy ROHF: -147.474399144
energy fci : -147.610218353


In [51]:
fci_rohf      = (fci_psi.dagger * hf_psi)
assert fci_rohf.imag == 0
fci_rohf = fci_rohf.real
fci_rohf_ovlp = np.abs(fci_rohf)**2

In [52]:
print(f" < FCI | ROHF >    = {fci_rohf:0.16f}")
print(f"|< FCI | ROHF >|^2 = {fci_rohf_ovlp:0.16f}")

 < FCI | ROHF >    = -0.0305582800991260
|< FCI | ROHF >|^2 = 0.0009338084826166


In [53]:
fci_sorted = fci_psi.sort() # <-- sort state by coefficent magnitude (largest first)
fci_sorted[:20] #<-- print top 20 terms for space

 0.297+0.000j |11111111011110111010> +
-0.297+0.000j |11111111101110111001> +
-0.297+0.000j |11111111011011101110> +
 0.297+0.000j |11111111101011101101> +
-0.258+0.000j |11111111101101111010> +
 0.258+0.000j |11111111101110110110> +
 0.258+0.000j |11111111100111101110> +
-0.258+0.000j |11111111101011011110> +
 0.217+0.000j |11111111011011111010> +
-0.217+0.000j |11111111101011111001> +
 0.217+0.000j |11111111011110101110> +
-0.217+0.000j |11111111101110101101> +
-0.189+0.000j |11111111100111111010> +
 0.189+0.000j |11111111101011110110> +
-0.189+0.000j |11111111101101101110> +
 0.189+0.000j |11111111101110011110> +
 0.119+0.000j |11111111111110111000> +
-0.119+0.000j |11111111111011101100> +
 0.087+0.000j |11111111111011111000> +
 0.087+0.000j |11111111111110101100>

In [54]:
def get_S2(state_mat):
    na = state_mat[0::2].sum()
    nb = state_mat[0::2].sum()
    ab = np.logical_and(state_mat[0::2], state_mat[1::2]).sum()
    a_minus_b_over_2 = (na-nb)/2
    S2_val = a_minus_b_over_2**2 + a_minus_b_over_2 + nb -ab
    return S2_val

In [55]:
print(f"S^2 HF: {get_S2(hf_psi.state_matrix[0])}")

S^2 HF: 2.0


In [56]:
# look over dominate ampliitude terms and measure energy

print_max = 4
counter = 0
for i in range(100):

    if counter>print_max:
        break

    S2_val = get_S2(fci_sorted[i].state_matrix[0])
    if not np.isclose(S2_val, 2):
        continue

    psi = QuantumState(fci_sorted[i].state_matrix, [1])
    print("psi:", psi)
    e_psi = H.expval(psi).real
    print(e_psi)
    psi_fci = psi.dagger * fci_psi
    assert psi_fci.imag ==0
    psi_fci = psi_fci.real

    print(f"S^2: {get_S2(psi.state_matrix[0])}")
    print(f" < psi | FCI > = {psi_fci:0.16f}")
    print(f"|< psi | FCI >|^2 = {np.abs(psi_fci)**2:0.16f}")
    print(f"e_psi - e_hf = {(e_psi-e_rohf):0.16f}")
    print()
    counter += 1

psi:  1.000+0.000j |11111111111110111000>
-146.96682032252866
S^2: 2.0
 < psi | FCI > = 0.1190479807066099
|< psi | FCI >|^2 = 0.0141724217103214
e_psi - e_hf = 0.5075788218669004

psi:  1.000+0.000j |11111111111011101100>
-146.96682017783166
S^2: 2.0
 < psi | FCI > = -0.1190195611246597
|< psi | FCI >|^2 = 0.0141656559303066
e_psi - e_hf = 0.5075789665638979

psi:  1.000+0.000j |11111111111011111000>
-146.96680982302743
S^2: 2.0
 < psi | FCI > = 0.0869353963855801
|< psi | FCI >|^2 = 0.0075577631447179
e_psi - e_hf = 0.5075893213681297

psi:  1.000+0.000j |11111111111110101100>
-146.9668099649968
S^2: 2.0
 < psi | FCI > = 0.0868544229115153
|< psi | FCI >|^2 = 0.0075436907792923
e_psi - e_hf = 0.5075891793987637

psi:  1.000+0.000j |11111111111111101000>
-147.4743991443956
S^2: 2.0
 < psi | FCI > = -0.0305582800991260
|< psi | FCI >|^2 = 0.0009338084826166
e_psi - e_hf = -0.0000000000000284



In [30]:
Hamiltonian_dictionary = H.to_dictionary

terms = list(Hamiltonian_dictionary.items())[:10] #<-- for space only print first 10 terms (use dictionary for access to all)
for P, coeff in terms:
    print(coeff, P)

(-91.92747732115532+0j) IIIIIIIIIIIIIIIIIIII
(12.772085205505254+0j) ZIIIIIIIIIIIIIIIIIII
(12.772085205505254+0j) IZIIIIIIIIIIIIIIIIII
(-1.3124086880046335e-05+0j) XZXIIIIIIIIIIIIIIIII
(-1.3124086880046335e-05+0j) YZYIIIIIIIIIIIIIIIII
(-1.3124086880046335e-05+0j) IXZXIIIIIIIIIIIIIIII
(-1.3124086880046335e-05+0j) IYZYIIIIIIIIIIIIIIII
(0.13040402808539597+0j) XZZZXIIIIIIIIIIIIIII
(0.13040402808539597+0j) YZZZYIIIIIIIIIIIIIII
(0.13040402808539597+0j) IXZZZXIIIIIIIIIIIIII
